# CSE465 ColorBench - Adaptive Skill Generation Pipeline

**Architecture:** Agentic Context Engineering (ACE Framework - ICLR 2026)
- **Self-Improving Agent:** Qwen2.5-VL-7B (4-bit NF4) acts as Generator, Reflector, and Solver.
- **Phase 1 (Skill Generator):** Generates visually-grounded cognitive skills.
- **Phase 2 (Skill Reflector):** Reflects on prediction errors and iteratively refines the playbook.
- **Phase 3 (Visual Solver):** Solves the question with the exact mandated prompt phrasing:
  *"This is the skill to solve this question, now solve the question and give me the answer."*

**Target:** Google Colab Free-Tier (Tesla T4 GPU, 15GB VRAM) — 100% Local Inference (0 API Keys)


## 1. Mount Google Drive


In [ ]:
from google.colab import drive
import os
from datetime import datetime

# Mount Drive
drive.mount('/content/drive')

# Set up Run Tag and Directory
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S") + "_ACE_Pipeline"
DRIVE_SAVE_DIR = f"/content/drive/MyDrive/CSE465_Results/{RUN_TAG}"

os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f"\nAll results will be saved to Google Drive: {DRIVE_SAVE_DIR}")


## 2. Install Dependencies


In [ ]:
!pip install -q transformers bitsandbytes accelerate qwen-vl-utils datasets "pillow<11.0.0"


## 3. Clone Repository


In [ ]:
import os

REPO_URL = "https://github.com/YOUR_USERNAME/cse465-project.git"  # <-- UPDATE THIS

# Go back to /content before cloning
%cd /content
if not os.path.exists("cse465-project"):
    !git clone {REPO_URL}

%cd cse465-project


## 4. Verify GPU


In [ ]:
import torch

# GPU Verification
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name} ({vram:.1f} GB VRAM) — Ready for 100% Local Execution.")
else:
    print("WARNING: No GPU detected! Set runtime type to GPU (T4).")


## 5. Load Models (Run Once)

Loads Qwen2.5-VL-7B in 4-bit precision (takes ~2 min).


In [ ]:
from qwen_solver import QwenSolver
from ace import ACE

# Load Qwen2.5-VL-7B in 4-bit (Unified Generator + Reflector + Solver)
solver = QwenSolver()
solver.load_model()

# Initialize ACE with the local solver (Self-Improving Agent)
ace_system = ACE(solver=solver)

print("\nAll models ready! Running 100% locally on Colab GPU.")


## 6. Select Task & Load Dataset (Interactive Form)

Use the dropdown below to select which ColorBench task to evaluate.


In [ ]:
#@title ⚙️ Task & Experiment Selection { run: "auto" }

SELECTED_TASK = "Color Illusion" #@param ["Color Recognition", "Color Illusion", "Color Mimicry", "Color Counting", "Color Comparison", "Color Blindness", "Object Counting", "Color Proportion", "Object Recognition", "Color Extraction"]
ITERATIONS = 2 #@param {type:"integer"}
QUICK_TEST = False #@param {type:"boolean"}
TEST_LIMIT = 5 #@param {type:"integer"}

from data_loader import ColorBenchDataLoader

TASK_SLUG = SELECTED_TASK.lower().replace(" ", "_")
loader = ColorBenchDataLoader(task_filter=SELECTED_TASK)
dataset_items = list(loader.stream_instances())
limit = TEST_LIMIT if QUICK_TEST else len(dataset_items)

print(f"\n{'='*75}")
print(f"SELECTED TASK: {SELECTED_TASK}")
print(f"Loaded {len(dataset_items)} total instances. Evaluating: {limit} instances.")
print(f"Iterations: {ITERATIONS} | Quick Test: {QUICK_TEST}")
print(f"{'='*75}\n")


## 7. Experiment: Baseline (Direct VQA, No Skills)


In [ ]:
import time
import os
import json
from data_loader import IncrementalLogger

# Save directly to Google Drive with dynamic task slug
baseline_filename = f"results_{TASK_SLUG}_baseline.jsonl"
baseline_path = os.path.join(DRIVE_SAVE_DIR, baseline_filename)
baseline_logger = IncrementalLogger(baseline_path)

correct, total = 0, 0
start = time.time()

print(f"\n{'='*75}")
print(f"STARTING BASELINE EVALUATION ({SELECTED_TASK} — {limit} questions)")
print(f"Logging to: {baseline_path}")
print(f"{'='*75}\n")

for item in dataset_items[:limit]:
    if item["idx"] in baseline_logger.processed_indices:
        continue

    q_start = time.time()
    labels = ['A', 'B', 'C', 'D', 'E', 'F']
    opt_str = ' | '.join(f'({labels[k]}) {c}' for k, c in enumerate(item['choices']))

    print(f"[{total + 1}/{limit}] Question (ID: {item['id']}): {item['question']}")
    print(f"Options: {opt_str}")

    result = solver.solve(item["image"], item["question"], item["choices"], mode="baseline")
    pred = result["prediction"]
    gt = item["answer"]
    is_correct = (pred == gt)
    if is_correct: correct += 1
    total += 1

    q_time = time.time() - q_start
    status = 'CORRECT' if is_correct else f'WRONG (Expected {gt})'
    print(f"  -> Model Output: {result['raw_output'][:150].strip()}")
    print(f"  -> Prediction: {pred} | GT: {gt} [{status}] (took {q_time:.1f}s)")
    print(f"  -> Running Score: {correct}/{total} ({correct/total*100:.1f}%)\n")

    baseline_logger.log_result({
        "idx": item["idx"], "id": item["id"], "task": item["task"],
        "question": item["question"], "choices": item["choices"],
        "ground_truth": gt, "prediction": pred, "is_correct": is_correct,
        "mode": "baseline", "raw_output": result["raw_output"]
    })

elapsed = time.time() - start
print(f"\n{'='*75}")
print(f"BASELINE COMPLETED ({SELECTED_TASK}): {correct}/{total} ({correct/total*100:.2f}%) in {elapsed:.1f}s")
print(f"Saved to: {baseline_path}")
print(f"{'='*75}\n")


## 8. Experiment: Adaptive Skill Generation (Iterative Prompting)


In [ ]:
import os
import time
import json
from data_loader import IncrementalLogger

# Save directly to Google Drive with dynamic task slug
skills_filename = f"results_{TASK_SLUG}_adaptive_skills_iter{ITERATIONS}.jsonl"
skills_path = os.path.join(DRIVE_SAVE_DIR, skills_filename)
skills_logger = IncrementalLogger(skills_path)

correct, total = 0, 0
start = time.time()

print(f"\n{'='*75}")
print(f"STARTING ADAPTIVE SKILL PIPELINE ({SELECTED_TASK} — {ITERATIONS} Iterations, {limit} questions)")
print(f"Logging to: {skills_path}")
print(f"{'='*75}\n")

for item in dataset_items[:limit]:
    if item["idx"] in skills_logger.processed_indices:
        continue

    q_start = time.time()
    question = item["question"]
    choices = item["choices"]
    image = item["image"]
    gt = item["answer"]

    labels = ['A', 'B', 'C', 'D', 'E', 'F']
    opt_str = ' | '.join(f'({labels[k]}) {c}' for k, c in enumerate(choices))

    print(f"[{total + 1}/{limit}] Question (ID: {item['id']}): {question}")
    print(f"Options: {opt_str}")

    # --- Iteration 1: Generate Visual Skill & Solve ---
    print("  [Iter 1] Formulating Visual Skill Directive...")
    plan = ace_system.generate_skill(question, choices, image=image)
    skill = plan.get("skill", "")
    classification = plan.get("classification", "unknown")
    print(f"  [Iter 1] Classification: {classification}")
    print(f"  [Iter 1] Generated Skill: {skill}")

    print("  [Iter 1] Solving Question with Qwen2.5-VL...")
    result = solver.solve(image, question, choices, mode="adaptive_skills", skill=skill)
    prediction = result["prediction"]
    raw_output = result["raw_output"]
    print(f"  [Iter 1] Model Raw Output: {raw_output[:120].strip()}...")
    print(f"  [Iter 1] Predicted Answer: {prediction}")

    all_iterations = [{"iter": 1, "skill": skill, "prediction": prediction}]

    # --- Iterations 2+: Reflect on Prediction & Refine Skill ---
    for i in range(2, ITERATIONS + 1):
        print(f"\n  [Iter {i}] Reflecting on Answer ({prediction}) & Refining Skill...")
        refined = ace_system.reflect_and_refine(
            question, choices, skill, prediction, raw_output, image=image
        )
        skill = refined.get("skill", skill)
        reflection = refined.get("reflection", "")
        print(f"  [Iter {i}] Reflection: {reflection}")
        print(f"  [Iter {i}] Refined Skill: {skill}")

        print(f"  [Iter {i}] Re-solving Question with Refined Skill...")
        result = solver.solve(image, question, choices, mode="adaptive_skills", skill=skill)
        prediction = result["prediction"]
        raw_output = result["raw_output"]
        print(f"  [Iter {i}] Model Raw Output: {raw_output[:120].strip()}...")
        print(f"  [Iter {i}] Predicted Answer: {prediction}")

        all_iterations.append({"iter": i, "skill": skill, "prediction": prediction, "reflection": reflection})

    is_correct = (prediction == gt)
    if is_correct: correct += 1
    total += 1

    q_time = time.time() - q_start
    status = 'CORRECT' if is_correct else f'WRONG (Expected {gt})'
    print(f"\n  -> Final Prediction: {prediction} | GT: {gt} [{status}] (took {q_time:.1f}s)")
    print(f"  -> Running Score: {correct}/{total} ({correct/total*100:.1f}%)\n")
    print(f"{'='*75}\n")

    skills_logger.log_result({
        "idx": item["idx"], "id": item["id"], "task": item["task"],
        "question": question, "choices": choices,
        "ground_truth": gt, "prediction": prediction, "is_correct": is_correct,
        "mode": "adaptive_skills", "classification": classification,
        "skill": skill, "iterations": all_iterations,
        "raw_output": raw_output
    })

elapsed = time.time() - start
print(f"\n{'='*75}")
print(f"ADAPTIVE SKILLS COMPLETED ({SELECTED_TASK} — {ITERATIONS} iters): {correct}/{total} ({correct/total*100:.2f}%) in {elapsed:.1f}s")
print(f"Saved to: {skills_path}")
print(f"{'='*75}\n")


## 9. Compare All Results

Evaluates and prints side-by-side comparative accuracy tables across all completed tasks.


In [ ]:
# Compare results stored in Google Drive
!python eval_results.py --dir "{DRIVE_SAVE_DIR}"
